In [221]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yanishelali/dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv


# Jour 2 — Phase 0 : Chargement des données

Objectif :
- récupérer le dataset Telco Customer Churn
- charger le fichier CSV
- vérifier que les données sont exploitables

Dataset :
- environ 7000 clients
- prédire le churn (résiliation)

In [222]:
import pandas as pd

In [223]:
# Charger le CSV
df = pd.read_csv("/kaggle/input/datasets/yanishelali/dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Vérification
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [224]:
print("Forme :", df.shape)
print(df.dtypes)

Forme : (7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


# Jour 2 — Phase 1 : Audit qualité

Objectif :
Analyser le dataset avant toute transformation :

- dimensions
- types des variables
- valeurs manquantes
- répartition de la cible (Churn)

Pourquoi ?
Un bon preprocessing commence toujours par un diagnostic.

In [225]:
def audit_qualite(df):
    """
    Affiche un rapport de santé du dataset :
    - forme (lignes, colonnes)
    - types des colonnes
    - % de valeurs manquantes
    - répartition de la cible Churn
    """

    print("=== FORMES DU DATASET ===")
    print(f"Lignes, colonnes : {df.shape}")

    print("\n=== TYPES DES COLONNES ===")
    print(df.dtypes)

    # Valeurs manquantes
    print("\n=== VALEURS MANQUANTES (%) ===")
    manquants = df.isna().mean() * 100
    manquants = manquants.sort_values(ascending=False)

    # Afficher seulement celles > 0
    print(manquants[manquants > 0].round(2))

    # Répartition de la cible
    print("\n=== RÉPARTITION DE LA CIBLE (Churn) ===")

    if "Churn" in df.columns:
        counts = df["Churn"].value_counts()
        total = len(df)

        for valeur, count in counts.items():
            pourcentage = (count / total) * 100
            print(f"{valeur} : {count} ({pourcentage:.2f}%)")
    else:
        print(" Colonne Churn introuvable")

In [226]:
audit_qualite(df)

=== FORMES DU DATASET ===
Lignes, colonnes : (7043, 21)

=== TYPES DES COLONNES ===
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

=== VALEURS MANQUANTES (%) ===
Series([], dtype: float64)

=== RÉPARTITION DE LA CIBLE (Churn) ===
No : 5174 (73.46%)
Yes : 1869 (26.54%)


In [227]:
print("=== TEST 1 : cas normal ===")
audit_qualite(df)

=== TEST 1 : cas normal ===
=== FORMES DU DATASET ===
Lignes, colonnes : (7043, 21)

=== TYPES DES COLONNES ===
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

=== VALEURS MANQUANTES (%) ===
Series([], dtype: float64)

=== RÉPARTITION DE LA CIBLE (Churn) ===
No : 5174 (73.46%)
Yes : 1869 (26.54%)


In [228]:
df_one_class = df[df["Churn"] == "No"]

print("\n=== TEST 2 : une seule classe ===")
audit_qualite(df_one_class)


=== TEST 2 : une seule classe ===
=== FORMES DU DATASET ===
Lignes, colonnes : (5174, 21)

=== TYPES DES COLONNES ===
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

=== VALEURS MANQUANTES (%) ===
Series([], dtype: float64)

=== RÉPARTITION DE LA CIBLE (Churn) ===
No : 5174 (100.00%)


In [229]:
print("\n=== TEST 3 : vérification déséquilibre ===")
audit_qualite(df)


=== TEST 3 : vérification déséquilibre ===
=== FORMES DU DATASET ===
Lignes, colonnes : (7043, 21)

=== TYPES DES COLONNES ===
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

=== VALEURS MANQUANTES (%) ===
Series([], dtype: float64)

=== RÉPARTITION DE LA CIBLE (Churn) ===
No : 5174 (73.46%)
Yes : 1869 (26.54%)


## Analyse

- Le dataset contient 7043 lignes et 21 colonnes.
- Plusieurs colonnes sont de type "object".
- Aucun manquant détecté directement.

Attention :
Certaines valeurs manquantes peuvent être cachées (ex: espaces dans TotalCharges).

- La cible est déséquilibrée (~73% / 27%)

Conclusion :
Le dataset nécessite un nettoyage approfondi avant toute modélisation.

# Phase 2 — Colonne piégée (TotalCharges)

Objectif :
- détecter les valeurs incohérentes
- convertir la colonne en numérique
- traiter les valeurs manquantes cachées

Problème :
La colonne TotalCharges est de type "object" alors qu'elle devrait être numérique.

In [230]:
print(df["TotalCharges"].head(20))

0       29.85
1      1889.5
2      108.15
3     1840.75
4      151.65
5       820.5
6      1949.4
7       301.9
8     3046.05
9     3487.95
10     587.45
11      326.8
12     5681.1
13     5036.3
14    2686.05
15    7895.15
16    1022.95
17    7382.25
18     528.35
19     1862.9
Name: TotalCharges, dtype: object


In [231]:
# Trouver les lignes suspectes
df[df["TotalCharges"] == " "]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [232]:
# Conversion en numérique
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [233]:
nb_nan = df["TotalCharges"].isna().sum()
print(f"Nombre de valeurs manquantes révélées : {nb_nan}")

Nombre de valeurs manquantes révélées : 11


In [234]:
# Imputation par la médiane
median = df["TotalCharges"].median()
df["TotalCharges"].fillna(median, inplace=True)

print("NaN restants :", df["TotalCharges"].isna().sum())

NaN restants : 0


/tmp/ipykernel_58/3170942738.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(median, inplace=True)


## Analyse

La colonne TotalCharges contenait des valeurs non numériques (espaces " "),
ce qui empêchait sa conversion en float.

Après conversion avec `pd.to_numeric(errors="coerce")` :
- 11 valeurs manquantes ont été révélées.

Choix effectué :
- imputation par la médiane

Justification :
- peu de valeurs manquantes (~0.15%)
- suppression aurait supprimé des données utiles
- la médiane est robuste aux valeurs aberrantes

In [235]:
print(df.dtypes["TotalCharges"])

float64


# Phase 3 — Encodage des variables

Objectif :
Transformer toutes les variables catégorielles en variables numériques.

Types de traitements :
- variables binaires → 0 / 1
- variables nominales → One-Hot Encoding
- suppression des identifiants inutiles

In [236]:
# Colonnes catégorielles
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Colonnes catégorielles :")
print(categorical_cols)

Colonnes catégorielles :
Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn'],
      dtype='object')


In [237]:
df = df.drop(columns=["customerID"])

In [238]:
# Mapping Yes / No → 1 / 0
binary_map = {"Yes": 1, "No": 0}

binary_cols = df.columns[df.nunique() == 2]

for col in binary_cols:
    df[col] = df[col].map(binary_map)

In [239]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,NaN,NaN,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,NaN,NaN,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,NaN,NaN,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,NaN,NaN,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,NaN,NaN,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [240]:
# One-Hot sur les colonnes restantes
df = pd.get_dummies(df, drop_first=True)

In [241]:
print(df.dtypes)

gender                                   float64
SeniorCitizen                            float64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No                          bool
OnlineSecurity_No internet service          bool
OnlineSecurity_Yes                          bool
OnlineBackup_No internet service            bool
OnlineBackup_Yes                            bool
DeviceProtection_No internet service        bool
DeviceProtection_Yes                        bool
TechSupport_No inter

In [242]:
print("Dataset final :", df.shape)

Dataset final : (7043, 31)


In [243]:
df.select_dtypes(include=["object"])

""
0
1
2
3
4
...
7038
7039
7040
7041


In [244]:
## Analyse

Les variables catégorielles ont été transformées :

- variables binaires (Yes/No) → 0/1
- variables nominales → One-Hot Encoding
- suppression de la colonne customerID

Résultat :
- dataset 100% numérique
- prêt pour un modèle de Machine Learning

Attention :
- le One-Hot Encoding augmente fortement le nombre de colonnes
- certaines variables rares peuvent créer beaucoup de dimensions

SyntaxError: invalid character '→' (U+2192) (3620583863.py, line 5)

# Phase 4 — Détection des valeurs aberrantes

Objectif :
- détecter les outliers
- analyser leur impact
- décider du traitement à appliquer

Méthode utilisée :


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in numerical_cols:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot de {col}")
    plt.show()

## Détection des outliers avec la règle IQR
``

In [ ]:
def detecter_outliers_iqr(df, colonne):
    Q1 = df[colonne].quantile(0.25)
    Q3 = df[colonne].quantile(0.75)
    IQR = Q3 - Q1

    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR

    outliers = df[(df[colonne] < borne_basse) | (df[colonne] > borne_haute)]

    print(f"\n--- {colonne} ---")
    print(f"Bornes : [{borne_basse:.2f}, {borne_haute:.2f}]")
    print(f"Nombre d'outliers : {len(outliers)}")

In [ ]:
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    detecter_outliers_iqr(df, col)

In [ ]:
def cap_outliers(df, colonne):
    Q1 = df[colonne].quantile(0.25)
    Q3 = df[colonne].quantile(0.75)
    IQR = Q3 - Q1

    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR

    df[colonne] = df[colonne].clip(borne_basse, borne_haute)

    return df

## Analyse

Des valeurs aberrantes ont été détectées dans plusieurs colonnes, notamment :

- TotalCharges
- MonthlyCharges

Cependant :

- ces valeurs élevées ne sont pas forcément des erreurs
- elles peuvent correspondre à de vrais clients (factures élevées)

Choix effectué :
- conservation des outliers

Justification :
- supprimer ces valeurs ferait perdre de l'information importante
- dans un contexte métier, les clients avec de fortes charges sont pertinents

Conclusion :
les outliers sont conservés.
